# Eval online: A/B, interleaving và drift trên Zalo Legal

Một bản retriever mới vừa deploy có lỗi cấu hình: câu hỏi bị cắt ở 15 từ đầu trước khi embed. Bạn đo xem lỗi đó bị phát
hiện nhanh tới đâu bằng ba cách: eval offline trên tập ghim, A/B test và interleaving với người dùng mô phỏng theo mô hình
click; rồi xem một thay đổi của **người dùng** (gõ không dấu) phá retrieval thế nào và phát hiện nó bằng drift.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Test chấm các hàm này trên ví dụ nhỏ.
- Cell `explore` là phần thiết kế của bạn: dựng thứ hạng, mô phỏng, vẽ đường power, đo drift, rồi viết `REPORT.md`.
- Cần Ollama với `bge-m3`, bộ `zalo-legal` và cache `bge-m3.npy` (module 13 bài Intermediate 01).
- Chấm: `uv run pytest tracks/ai-engineer/modules/20-evaluation-production/exercises/advanced/01-online-eval -q`

In [ ]:
# explore: đưa gốc repo vào sys.path, tìm thư mục dữ liệu
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
ZALO = ROOT / "data" / "raw" / "zalo-legal"

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")

import math
from statistics import NormalDist

import numpy as np

## 1. Các hàm được chấm

`team_draft_interleave(a, b, k, rng) -> (merged, team)`: `a`, `b` là hai danh sách id đã xếp hạng, `rng` là
`np.random.Generator`. Lặp tới khi `merged` đủ `k` phần tử hoặc cả hai danh sách hết phần tử mới: đội có **ít** phần tử
trong `merged` hơn được chọn; bằng nhau thì `rng.random() < 0.5` chọn A, ngược lại B (chỉ gọi `rng` trong trường hợp
bằng nhau). Đội được chọn thêm phần tử xếp hạng cao nhất **chưa có** trong `merged` của danh sách của mình; nếu danh sách
đó không còn phần tử mới thì đội kia chọn. `team` là dict `id -> "A"` hoặc `"B"`.

`interleave_outcome(clicked, team) -> str`: đếm click thuộc đội A và đội B (id không có trong `team` bỏ qua); nhiều hơn
thì thắng, bằng nhau là `"tie"`.

`cascade_clicks(ranking, relevant, rng, p_rel=0.9, p_nonrel=0.1, p_stop=0.8) -> list`: người dùng xem từ trên xuống. Với
mỗi id theo thứ tự: gọi `u = rng.random()`, click nếu `u < p_rel` (id thuộc `relevant`) hoặc `u < p_nonrel` (không thuộc).
Nếu vừa click một id **liên quan**, gọi thêm `v = rng.random()`, và dừng xem nếu `v < p_stop`. Trả các id đã click theo thứ
tự. Không gọi `rng` thêm lần nào khác.

`ab_sample_size(p_a, p_b, alpha=0.05, power=0.8) -> int`: số mẫu **mỗi nhóm** theo công thức ở mục 1 lý thuyết (z hai
phía), làm tròn lên. `ValueError` khi `p_a == p_b` hoặc tỉ lệ ngoài (0, 1).

`sign_test_pvalue(wins_a, wins_b) -> float`: p-value hai phía của kiểm định dấu (binomial với p = 0.5) trên các lượt có
người thắng: $$\min\left(1, 2\sum_{i \le \min(w_A, w_B)} \binom{n}{i} 2^{-n}\right)$$ với $$n = w_A + w_B$$; $$n = 0$$ trả 1.0.

`psi(expected, actual, edges) -> float`: tỉ lệ mẫu trong từng bin của `np.histogram(..., bins=edges)` cho mỗi dãy, kẹp
dưới ở `1e-4` (để không chia cho 0), rồi $$\sum_b (a_b - e_b)\ln(a_b/e_b)$$.

In [ ]:
def team_draft_interleave(a: list, b: list, k: int, rng) -> tuple[list, dict]:
    # TODO
    raise NotImplementedError


def interleave_outcome(clicked: list, team: dict) -> str:
    # TODO
    raise NotImplementedError


def cascade_clicks(ranking: list, relevant: set, rng, p_rel: float = 0.9, p_nonrel: float = 0.1,
                   p_stop: float = 0.8) -> list:
    # TODO
    raise NotImplementedError


def ab_sample_size(p_a: float, p_b: float, alpha: float = 0.05, power: float = 0.8) -> int:
    # TODO
    raise NotImplementedError


def sign_test_pvalue(wins_a: int, wins_b: int) -> float:
    # TODO
    raise NotImplementedError


def psi(expected, actual, edges) -> float:
    # TODO
    raise NotImplementedError

## 2. Thiết kế của bạn

Gợi ý các bước (thứ tự và cách làm là của bạn):

1. **Thứ hạng.** 300 câu đầu của `qrels/test.jsonl` (tập ghim của module 13). Hệ thống A: dense `bge-m3` trên câu hỏi đầy
   đủ, top-10 trên toàn corpus (cache `bge-m3.npy`, nhớ chuẩn hoá). Hệ thống B: cùng retriever nhưng câu hỏi bị cắt còn 15
   từ đầu (`" ".join(q.split()[:15])`). Lưu thứ hạng ra đĩa.
2. **Offline.** MRR@10 và hit@1 của A và B, bootstrap **theo cặp** cho hiệu MRR (module 11 mục 7), số câu có thứ hạng đổi.
3. **Online mô phỏng.** Người dùng là một câu hỏi lấy ngẫu nhiên có hoàn lại từ 300 câu, click theo `cascade_clicks` với
   tham số mặc định; "thành công" là có click vào một điều luật đúng. A/B: nửa người dùng thấy A, nửa thấy B, kiểm định hai
   tỉ lệ. Interleaving: mọi người dùng thấy danh sách team-draft 10 phần tử, kiểm định dấu trên số lượt A thắng, B thắng.
   Với mỗi tổng số người dùng $$N$$ trong một lưới (100 tới 6 400), lặp 300 thí nghiệm, power là tỉ lệ thí nghiệm phát hiện
   "A tốt hơn" ở mức 0.05. So với `ab_sample_size` tính từ tỉ lệ thành công thật (ước bằng mô phỏng rất lớn).
4. **Drift.** Embed lại 300 câu hỏi ở dạng **không dấu** (bỏ dấu thanh và đổi đ thành d), đo MRR@10 của retriever. Chọn một
   hai đặc trưng rẻ tính được trên từng câu hỏi (ví dụ tỉ lệ ký tự có dấu, số từ) và tính PSI giữa: câu hỏi train và câu hỏi
   test (không drift), câu hỏi test và bản không dấu (drift). Đặt ngưỡng cảnh báo của bạn.
5. Viết `REPORT.md` theo khung, làm trắc nghiệm của bài trong app.

Mô phỏng 300 thí nghiệm cho 7 mức $$N$$ chạy cỡ vài phút trên CPU bằng Python thuần; vector hoá nếu muốn nhanh hơn.

In [ ]:
# explore: thiết kế của bạn ở đây (dựng thứ hạng, offline, mô phỏng, drift)